In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer

class LinearRegressionScratch:
    def __init__(self, learning_rate=0.05, epochs=500):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None
        self.loss_history = []

    def f(self, X):
        return np.dot(X, self.w)

    def compute_loss(self, X, y):
        N = len(y)
        return (1 / (2 * N)) * np.sum((self.f(X) - y) ** 2)

    def derivative(self, X, y):
        N = len(y)
        return (1 / N) * np.dot(X.T, (self.f(X) - y))

    def fit(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        n_samples, n_features = X_b.shape
        self.w = np.zeros(n_features)
        self.loss_history = []

        for _ in range(self.epochs):
            loss = self.compute_loss(X_b, y)
            self.loss_history.append(loss)
            grad = self.derivative(X_b, y)
            self.w -= self.learning_rate * grad

    def predict(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        return self.f(X_b)

    @staticmethod
    def evaluate(y_true, y_pred):
        mse = np.mean((y_true - y_pred) ** 2)
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        return {"MSE": mse, "R2_Score": r2}


class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.1, epochs=500):
        self.alpha = learning_rate
        self.epochs = epochs
        self.theta = None
        self.loss_history = []

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def h(self, X):
        return self.sigmoid(np.dot(X, self.theta))

    def compute_loss(self, X, y):
        m = len(y)
        h_val = np.clip(self.h(X), 1e-15, 1 - 1e-15)
        return - (1 / m) * np.sum(y * np.log(h_val) + (1 - y) * np.log(1 - h_val))

    def derivative(self, X, y):
        m = len(y)
        return (1 / m) * np.dot(X.T, (self.h(X) - y))

    def fit(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        m, n = X_b.shape
        self.theta = np.zeros(n)
        self.loss_history = []

        for _ in range(self.epochs):
            loss = self.compute_loss(X_b, y)
            self.loss_history.append(loss)
            grad = self.derivative(X_b, y)
            self.theta -= self.alpha * grad

    def predict_proba(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        return self.h(X_b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    @staticmethod
    def evaluate(y_true, y_pred):
        accuracy = np.mean(y_true == y_pred)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1_Score": f1}

In [ ]:
np.random.seed(42)

# 1. Linear Regression on Diabetes Dataset (UCI Benchmark)
X_lin, y_lin = load_diabetes(return_X_y=True)
X_lin = (X_lin - np.mean(X_lin, axis=0)) / np.std(X_lin, axis=0)

indices = np.random.permutation(len(X_lin))
split = int(0.8 * len(X_lin))
X_train_lin, X_test_lin = X_lin[indices[:split]], X_lin[indices[split:]]
y_train_lin, y_test_lin = y_lin[indices[:split]], y_lin[indices[split:]]

lin_model = LinearRegressionScratch(learning_rate=0.05, epochs=500)
lin_model.fit(X_train_lin, y_train_lin)

y_pred_lin = lin_model.predict(X_test_lin)
metrics_lin = lin_model.evaluate(y_test_lin, y_pred_lin)
print("Linear Regression (Diabetes Dataset):")
print(f"Test MSE: {metrics_lin['MSE']:.4f}, R2: {metrics_lin['R2_Score']:.4f}")

# 2. Logistic Regression on Breast Cancer Wisconsin (UCI Benchmark)
X_log, y_log = load_breast_cancer(return_X_y=True)
X_log = (X_log - np.mean(X_log, axis=0)) / np.std(X_log, axis=0)

indices = np.random.permutation(len(X_log))
split = int(0.8 * len(X_log))
X_train_log, X_test_log = X_log[indices[:split]], X_log[indices[split:]]
y_train_log, y_test_log = y_log[indices[:split]], y_log[indices[split:]]

log_model = LogisticRegressionScratch(learning_rate=0.1, epochs=500)
log_model.fit(X_train_log, y_train_log)

y_pred_log = log_model.predict(X_test_log)
metrics_log = log_model.evaluate(y_test_log, y_pred_log)
print("\nLogistic Regression (Breast Cancer Dataset):")
print("Test Evaluation:", {k: round(v, 4) for k, v in metrics_log.items()})

# 3. Plot Training Loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(lin_model.loss_history, color='blue', lw=2)
plt.title('Linear Regression - Diabetes Dataset')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(log_model.loss_history, color='red', lw=2)
plt.title('Logistic Regression - Breast Cancer Dataset')
plt.xlabel('Epochs')
plt.ylabel('Loss (Cross-Entropy)')
plt.grid(True)

plt.tight_layout()
plt.show()